In [1]:
# ==============================================================================
# main.py: Train EN-BN and EN-HI Translation Models (Full Data) & Generate Submission
#
# ----- ARCHITECTURE: TRANSFORMER (from scratch w/ PyTorch layers) -----
# ==============================================================================
print("Starting the combined English-to-Indian Language NMT Pipeline.")
print("Using TRANSFORMER architecture. Training on full dataset.")
print("-" * 70)

# ==============================================================================
# Step 1: Installations and Imports
# ==============================================================================
print("Step 1: Importing libraries...")
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import json
import pandas as pd
from tqdm import tqdm
import random
import os
import gc
import torch.nn.functional as F
import math
import heapq

# For building our subword tokenizers
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("-" * 70)

# ==============================================================================
# Step 2: Load Full Training Data for Both Language Pairs
# ==============================================================================
print("Step 2: Loading full training data...")

def load_json_data(json_path):
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"Error: '{json_path}' not found.")
        exit()
    except Exception as e:
        print(f"Error loading {json_path}: {e}")
        exit()

train_data_path = '/kaggle/input/capstone/train_data1.json'
print(f"Loading training data from '{train_data_path}'...")
data = load_json_data(train_data_path)

# --- Process English-Bengali Data ---
en_train_bn_pairs = []
try:
    for entry_id, entry_data in data["English-Bengali"]["Train"].items():
        en_train_bn_pairs.append((entry_data["source"], entry_data["target"]))
    source_sentences_train_bn, target_sentences_train_bn = zip(*en_train_bn_pairs)
    print(f"English-Bengali: Loaded {len(source_sentences_train_bn)} training pairs.")
except KeyError as e:
    print(f"Error: Missing key {e} in English-Bengali data.")
    exit()

# --- Process English-Hindi Data ---
en_train_hi_pairs = []
try:
    for entry_id, entry_data in data["English-Hindi"]["Train"].items():
        en_train_hi_pairs.append((entry_data["source"], entry_data["target"]))
    source_sentences_train_hi, target_sentences_train_hi = zip(*en_train_hi_pairs)
    print(f"English-Hindi: Loaded {len(source_sentences_train_hi)} training pairs.")
except KeyError as e:
    print(f"Error: Missing key {e} in English-Hindi data.")
    exit()

all_en_train_sentences = list(source_sentences_train_bn) + list(source_sentences_train_hi)
print(f"Total English sentences for tokenizer training: {len(all_en_train_sentences)}")
print("-" * 70)

# ==============================================================================
# Step 3: Train Subword Tokenizers (EN, BN, HI)
# ==============================================================================
print("Step 3: Training subword tokenizers...")

def train_tokenizer(sentences, vocab_size, filename_prefix, lang_name):
    tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()
    trainer = WordPieceTrainer(vocab_size=vocab_size, special_tokens=["[PAD]", "[UNK]", "[SOS]", "[EOS]"])
    tokenizer.train_from_iterator(sentences, trainer=trainer)
    filename = f"{filename_prefix}.json"
    tokenizer.save(filename)
    print(f"{lang_name} tokenizer trained ({vocab_size} vocab) and saved to '{filename}'.")
    return tokenizer

EN_VOCAB_SIZE = 16000
BN_VOCAB_SIZE = 20000
HI_VOCAB_SIZE = 20000

src_tokenizer = train_tokenizer(all_en_train_sentences, EN_VOCAB_SIZE, "src_tokenizer_en_combined", "English (Combined)")
tgt_tokenizer_bn = train_tokenizer(target_sentences_train_bn, BN_VOCAB_SIZE, "tgt_tokenizer_bn", "Bengali")
tgt_tokenizer_hi = train_tokenizer(target_sentences_train_hi, HI_VOCAB_SIZE, "tgt_tokenizer_hi", "Hindi")

src_tokenizer = Tokenizer.from_file("src_tokenizer_en_combined.json")
tgt_tokenizer_bn = Tokenizer.from_file("tgt_tokenizer_bn.json")
tgt_tokenizer_hi = Tokenizer.from_file("tgt_tokenizer_hi.json")
print("Tokenizers reloaded.")
print("-" * 70)

# ==============================================================================
# Step 4: Create PyTorch Dataset and Define TRANSFORMER Model
# ==============================================================================
print("Step 4: Defining Dataset and TRANSFORMER Model Architecture...")
MAX_LENGTH = 50 # Max sequence length

class TranslationDataset(Dataset):
    def __init__(self, source_sentences, target_sentences, src_tokenizer, tgt_tokenizer, max_length):
        self.source_sentences, self.target_sentences = source_sentences, target_sentences
        self.src_tokenizer, self.tgt_tokenizer = src_tokenizer, tgt_tokenizer
        self.max_length = max_length
        self.pad_id = self.src_tokenizer.token_to_id("[PAD]")
        self.sos_id = self.src_tokenizer.token_to_id("[SOS]")
        self.eos_id = self.src_tokenizer.token_to_id("[EOS]")

    def __len__(self):
        return len(self.source_sentences)

    def __getitem__(self, idx):
        src_sentence = self.source_sentences[idx]; tgt_sentence = self.target_sentences[idx]
        src_encoding = self.src_tokenizer.encode(src_sentence)
        tgt_encoding = self.tgt_tokenizer.encode(tgt_sentence)

        # Truncate if longer than max_length - 2 (for SOS/EOS)
        src_ids = src_encoding.ids[:self.max_length-2]
        tgt_ids = tgt_encoding.ids[:self.max_length-2]

        # --- MODIFICATION for Transformer ---
        # 1. Source sequence
        src = [self.sos_id] + src_ids + [self.eos_id]
        
        # 2. Target input (starts with SOS)
        tgt_input = [self.sos_id] + tgt_ids
        
        # 3. Target output (ends with EOS)
        tgt_output = tgt_ids + [self.eos_id]

        # Pad all three to max_length
        src_padding = [self.pad_id] * (self.max_length - len(src))
        tgt_input_padding = [self.pad_id] * (self.max_length - len(tgt_input))
        tgt_output_padding = [self.pad_id] * (self.max_length - len(tgt_output))
        
        return {
            "src": torch.tensor(src + src_padding, dtype=torch.long),
            "tgt_input": torch.tensor(tgt_input + tgt_input_padding, dtype=torch.long),
            "tgt_output": torch.tensor(tgt_output + tgt_output_padding, dtype=torch.long)
        }

# --- NEW: Positional Encoding ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # Shape: [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [batch_size, seq_len, d_model]
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# --- NEW: Transformer Model ---
class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, nhead, num_encoder_layers,
                 num_decoder_layers, dim_feedforward, dropout=0.1, max_len=50):
        super(Seq2SeqTransformer, self).__init__()
        self.d_model = d_model
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout, max_len)
        
        # Using the built-in PyTorch Transformer (compliant with rules)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True  # We use batch_first=True
        )
        
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        self.pad_id = 0 # Assuming PAD ID is 0

    def generate_square_subsequent_mask(self, sz):
        # Generates a mask that prevents the decoder from looking at future tokens
        return torch.triu(torch.ones(sz, sz) * float('-inf'), diagonal=1)

    def create_padding_mask(self, seq):
        # seq shape: [batch_size, seq_len]
        # Returns mask: [batch_size, seq_len]
        return (seq == self.pad_id)

    def forward(self, src, tgt_input):
        # src shape: [batch_size, src_seq_len]
        # tgt_input shape: [batch_size, tgt_seq_len]
        
        # 1. Create masks
        src_padding_mask = self.create_padding_mask(src).to(device) # [batch_size, src_seq_len]
        tgt_padding_mask = self.create_padding_mask(tgt_input).to(device) # [batch_size, tgt_seq_len]
        
        tgt_subsequent_mask = self.generate_square_subsequent_mask(tgt_input.size(1)).to(device) # [tgt_seq_len, tgt_seq_len]
        
        # 2. Embed and add positional encoding
        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.tgt_embedding(tgt_input) * math.sqrt(self.d_model))
        
        # 3. Pass through Transformer
        output = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_subsequent_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )
        
        # 4. Final output projection
        return self.fc_out(output)

print("Dataset and Transformer Model defined.")
print("-" * 70)

# ==============================================================================
# Step 5: Training Function and "Noam" LR Scheduler
# ==============================================================================
print("Step 5: Defining Training function and Noam Scheduler...")

# --- CRITICAL: Noam Scheduler for Transformers ---
class NoamLR(optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, d_model, warmup_steps, last_epoch=-1):
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        super().__init__(optimizer, last_epoch)
    
    def get_lr(self):
        step = self.last_epoch + 1
        scale = (self.d_model**-0.5) * min(step**-0.5, step * self.warmup_steps**-1.5)
        return [base_lr * scale for base_lr in self.base_lrs]

def init_weights(m):
    # Standard Transformer initialization
    if hasattr(m, 'weight') and m.weight.dim() > 1:
        nn.init.xavier_uniform_(m.weight.data)
        
print("Training function defined.")
print("-" * 70)

# ==============================================================================
# Step 6: Train EN-BN and EN-HI Models Separately (Full Data)
# ==============================================================================
print("Step 6: Training models on full dataset...")

def run_training_for_pair(lang_pair_name, source_train, target_train, src_tok, tgt_tok, model_save_path):
    print(f"\n--- Training {lang_pair_name} ---")

    # --- Use a custom collate_fn for the Transformer dataset ---
    def collate_fn_transformer(batch):
        src_batch, tgt_input_batch, tgt_output_batch = [], [], []
        for item in batch:
            src_batch.append(item["src"])
            tgt_input_batch.append(item["tgt_input"])
            tgt_output_batch.append(item["tgt_output"])
        
        # pad_sequence expects a list of tensors, and batch_first=True
        return {
            "src": torch.stack(src_batch),
            "tgt_input": torch.stack(tgt_input_batch),
            "tgt_output": torch.stack(tgt_output_batch)
        }

    train_dataset = TranslationDataset(source_train, target_train, src_tok, tgt_tok, MAX_LENGTH)
    # Note: collate_fn is now used
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                                  num_workers=2, pin_memory=True, collate_fn=collate_fn_transformer)

    # --- Initialize Transformer Model ---
    INPUT_DIM, OUTPUT_DIM = src_tok.get_vocab_size(), tgt_tok.get_vocab_size()
    
    model = Seq2SeqTransformer(
        src_vocab_size=INPUT_DIM,
        tgt_vocab_size=OUTPUT_DIM,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_encoder_layers=NUM_LAYERS,
        num_decoder_layers=NUM_LAYERS,
        dim_feedforward=DIM_FF,
        dropout=DROPOUT,
        max_len=MAX_LENGTH
    ).to(device)
    
    model.apply(init_weights) # Apply Xavier initialization

    # --- Optimizer, Scheduler, and Loss ---
    # Base LR 1.0 is used because NoamLR handles the scaling
    optimizer = optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
    scheduler = NoamLR(optimizer, d_model=D_MODEL, warmup_steps=WARMUP_STEPS)
    
    TGT_PAD_IDX = tgt_tok.token_to_id('[PAD]')
    criterion = nn.CrossEntropyLoss(ignore_index=TGT_PAD_IDX)

    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    # --- Training Loop (modified for Transformer) ---
    for epoch in range(N_EPOCHS):
        model.train()
        epoch_loss = 0
        progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{N_EPOCHS} Training", leave=False)
        
        for i, batch in enumerate(progress_bar):
            # Data is already [batch_size, seq_len]
            src = batch['src'].to(device)
            tgt_input = batch['tgt_input'].to(device)
            tgt_output = batch['tgt_output'].to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            output = model(src, tgt_input)
            # output shape: [batch_size, seq_len, output_dim]
            
            # Loss calculation
            output_dim = output.shape[-1]
            # Flatten the batch for loss
            loss = criterion(output.reshape(-1, output_dim), tgt_output.reshape(-1))
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            optimizer.step()
            scheduler.step() # Update learning rate
            
            epoch_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_lr()[0]:.7f}'})
        
        print(f'Epoch: {epoch+1:02} | Train Loss: {epoch_loss / len(train_dataloader):.3f}')

    torch.save(model.state_dict(), model_save_path)
    print(f"{lang_pair_name} model saved to '{model_save_path}'.")

    del model, optimizer, criterion, train_dataset, train_dataloader, scheduler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# --- Training Parameters for TRANSFORMER ---
# Matching your friend's setup
D_MODEL = 256      # This is the "hid_dim" for Transformers
N_HEAD = 8         # Number of attention heads (512 must be divisible by 8)
NUM_LAYERS = 1     # 6 layers for encoder, 6 for decoder (Standard "Transformer-Base")
DIM_FF = 2048      # Feedforward layer size (4 * d_model)
DROPOUT = 0.1
CLIP = 1.0
N_EPOCHS = 30      # Matching your friend's 20-25
BATCH_SIZE = 256
WARMUP_STEPS = 4000 # Standard for "Noam" scheduler

# --- Train English-Bengali ---
run_training_for_pair(
    "English-Bengali",
    source_sentences_train_bn, target_sentences_train_bn,
    src_tokenizer, tgt_tokenizer_bn,
    "nmt-model-transformer-bengali-full.pt"
)

# --- Train English-Hindi ---
run_training_for_pair(
    "English-Hindi",
    source_sentences_train_hi, target_sentences_train_hi,
    src_tokenizer, tgt_tokenizer_hi,
    "nmt-model-transformer-hindi-full.pt"
)

print("\nAll models (trained on full data) saved successfully.")
print("-" * 70)

# ==============================================================================
# Step 7: Final Submission Generation (Updated for Transformer)
# ==============================================================================
print("Step 7: Generating final Codabench submission file...")

# --- NEW Inference Function (Greedy Decoding for Transformer) ---
# --- Inference Function (FIXED with input truncation) ---
def translate_sentence(sentence, src_tokenizer, tgt_tokenizer, model, device, max_len=50):
    model.eval()
    
    # 1. Tokenize source
    src_encoding = src_tokenizer.encode(sentence)
    
    # --- THIS IS THE FIX ---
    # Truncate the source tokens just like in the dataset
    src_ids = src_encoding.ids[:MAX_LENGTH-2] # Truncate to 48 tokens
    # --- END OF FIX ---
    
    src_indexes = ([src_tokenizer.token_to_id("[SOS]")] +
                   src_ids +  # Use the truncated src_ids
                   [src_tokenizer.token_to_id("[EOS]")])
    
    # Pad the source tensor *manually* to MAX_LENGTH
    # (This isn't strictly necessary for a single sentence, but good practice)
    src_padding = [src_tokenizer.token_to_id("[PAD]")] * (MAX_LENGTH - len(src_indexes))
    src_tensor = torch.LongTensor(src_indexes + src_padding).unsqueeze(0).to(device) # [1, MAX_LENGTH]
    
    # 2. Create source mask
    src_padding_mask = model.create_padding_mask(src_tensor).to(device) # [1, 1, MAX_LENGTH]
    
    # 3. Get Encoder output (memory)
    with torch.no_grad():
        src_emb = model.pos_encoder(model.src_embedding(src_tensor) * math.sqrt(model.d_model))
        memory = model.transformer.encoder(src_emb, src_key_padding_mask=src_padding_mask)
        # memory shape: [1, src_len, d_model]

    # 4. Start decoding loop
    tgt_indexes = [tgt_tokenizer.token_to_id('[SOS]')]
    
    for i in range(max_len):
        # Create target tensor from current sequence
        tgt_tensor = torch.LongTensor(tgt_indexes).unsqueeze(0).to(device) # [1, current_tgt_len]
        
        # Create masks for decoder
        tgt_padding_mask = model.create_padding_mask(tgt_tensor).to(device)
        tgt_subsequent_mask = model.generate_square_subsequent_mask(tgt_tensor.size(1)).to(device)

        with torch.no_grad():
            # Embed target
            tgt_emb = model.pos_encoder(model.tgt_embedding(tgt_tensor) * math.sqrt(model.d_model))
            # Get decoder output
            output = model.transformer.decoder(
                tgt_emb,
                memory,
                tgt_mask=tgt_subsequent_mask,
                tgt_key_padding_mask=tgt_padding_mask,
                memory_key_padding_mask=src_padding_mask # Use the same src mask
            )
            
            # Project to vocabulary
            logits = model.fc_out(output[:, -1, :]) # Get logits for ONLY the last token
        
        # Get the predicted token (greedy)
        pred_token = logits.argmax(1).item()
        tgt_indexes.append(pred_token)

        # 5. Stop if <eos> token is predicted
        if pred_token == tgt_tokenizer.token_to_id('[EOS]'):
            break

    # 6. Decode token indexes back to string
    trg_tokens = tgt_tokenizer.decode(tgt_indexes[1:], skip_special_tokens=True) # Skip <sos>
    return trg_tokens

# --- Load Official Validation Data ---
val_data_path = '/kaggle/input/test-data/test_data1_final.json'
print(f"Loading official validation data from '{val_data_path}'...")
codabench_val_data = load_json_data(val_data_path)

en_bn_val_pairs_official = []
en_hi_val_pairs_official = []
try:
    for entry_id, entry_data in codabench_val_data["English-Bengali"]["Test"].items():
        en_bn_val_pairs_official.append((entry_id, entry_data["source"]))
    for entry_id, entry_data in codabench_val_data["English-Hindi"]["Test"].items():
        en_hi_val_pairs_official.append((entry_id, entry_data["source"]))
    print("Official test sentences extracted.")
except KeyError as e:
     print(f"Error: Missing key {e} in '{val_data_path}'. Cannot generate submission.")
     exit()

# --- Submission Generation Function ---
def generate_submission_for_pair(lang_pair_name, official_val_pairs, src_tok, tgt_tok, model_path):
    print(f"\n--- Generating {lang_pair_name} submissions ---")

    # --- MODIFICATION: Instantiate TRANSFORMER model ---
    INPUT_DIM, OUTPUT_DIM = src_tok.get_vocab_size(), tgt_tok.get_vocab_size()

    model = Seq2SeqTransformer(
        src_vocab_size=INPUT_DIM,
        tgt_vocab_size=OUTPUT_DIM,
        d_model=D_MODEL,
        nhead=N_HEAD,
        num_encoder_layers=NUM_LAYERS,
        num_decoder_layers=NUM_LAYERS,
        dim_feedforward=DIM_FF,
        dropout=DROPOUT,
        max_len=MAX_LENGTH
    ).to(device)
    # ----------------------------------------------------

    try:
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"Loaded model state from '{model_path}'")
    except FileNotFoundError:
        print(f"Error: Model file '{model_path}' not found. Cannot generate translations.")
        return []
    except Exception as e:
        print(f"Error loading model {model_path}: {e}. Cannot generate translations.")
        return []

    translations = []
    for entry_id, sentence in tqdm(official_val_pairs, desc=f"Translating {lang_pair_name} Codabench Set", leave=False):
        # Use the new Transformer-based translate_sentence
        translation = translate_sentence(sentence, src_tok, tgt_tok, model, device, max_len=MAX_LENGTH)
        translations.append({'ID': entry_id, 'Translation': translation})

    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    print(f"Generated {len(translations)} translations for {lang_pair_name}.")
    return translations

# --- Generate Translations for Both Pairs ---
all_translations_submission = []

bn_submissions = generate_submission_for_pair(
    "English-Bengali",
    en_bn_val_pairs_official,
    src_tokenizer, tgt_tokenizer_bn,
    "nmt-model-transformer-bengali-full.pt" # Use new Transformer model file
)
all_translations_submission.extend(bn_submissions)

hi_submissions = generate_submission_for_pair(
    "English-Hindi",
    en_hi_val_pairs_official,
    src_tokenizer, tgt_tokenizer_hi,
    "nmt-model-transformer-hindi-full.pt" # Use new Transformer model file
)
all_translations_submission.extend(hi_submissions)

# --- Save Combined Submission File ---
submission_filename = 'answer.csv'
print(f"\nWriting combined submission file: {submission_filename}...")
try:
    with open(submission_filename, 'w', encoding='utf-8') as f:
        f.write("ID\tTranslation\n") # Header
        for item in all_translations_submission:
            translation_text = str(item['Translation']).replace('"', '""') # Escape quotes
            f.write(f"{item['ID']}\t\"{translation_text}\"\n") # Row with quotes

    print("\n" + "="*50)
    print(f"✓ Final submission file '{submission_filename}' created successfully!")
    print(f"   Format: ID<tab>\"Translation\"")
    print(f"Total translations: {len(all_translations_submission)}")
    print("="*50)

    print("\n--- File Preview (First 5 lines) ---")
    with open(submission_filename, 'r', encoding='utf-8') as f_check:
        for i, line in enumerate(f_check):
            if i >= 5: break
            print(line.strip())
except Exception as e:
    print(f"\nError writing submission file: {e}")

print("\nPipeline finished successfully.")

Starting the combined English-to-Indian Language NMT Pipeline.
Using TRANSFORMER architecture. Training on full dataset.
----------------------------------------------------------------------
Step 1: Importing libraries...
Using device: cuda
----------------------------------------------------------------------
Step 2: Loading full training data...
Loading training data from '/kaggle/input/capstone/train_data1.json'...
English-Bengali: Loaded 68849 training pairs.
English-Hindi: Loaded 80797 training pairs.
Total English sentences for tokenizer training: 149646
----------------------------------------------------------------------
Step 3: Training subword tokenizers...



English (Combined) tokenizer trained (16000 vocab) and saved to 'src_tokenizer_en_combined.json'.



Bengali tokenizer trained (20000 vocab) and saved to 'tgt_tokenizer_bn.json'.



Hindi tokenizer trained (20000 vocab) and saved to 'tgt_tokenizer_hi.json'.
Tokenizers reloaded.
----------------------------------------

Epoch 1/30 Training:   0%|          | 0/269 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/usr/local/lib/python3.11/dist-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Epoch: 01 | Train Loss: 9.306


Epoch 2/30 Training:   0%|          | 0/269 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch: 02 | Train Loss: 7.830


...

Epoch 30/30 Training:   0%|          | 0/316 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch: 30 | Train Loss: 1.067
English-Hindi model saved to 'nmt-model-transformer-hindi-full.pt'.

All models (trained on full data) saved successfully.
----------------------------------------------------------------------
Step 7: Generating final Codabench submission file...
Loading official validation data from '/kaggle/input/test-data/test_data1_final.json'...
Official test sentences extracted.

--- Generating English-Bengali submissions ---
Loaded model state from 'nmt-model-transformer-bengali-full.pt'


Translating English-Bengali Codabench Set:   0%|          | 0/19672 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Generated 19672 translations for English-Bengali.

--- Generating English-Hindi submissions ---
Loaded model state from 'nmt-model-transformer-hindi-full.pt'


Generated 23085 translations for English-Hindi.

Writing combined submission file: answer.csv...

✓ Final submission file 'answer.csv' created successfully!
   Format: ID<tab>"Translation"
Total translations: 42757

--- File Preview (First 5 lines) ---
ID	Translation
177039	"বর্তমান ঘটনা"
177040	"ভগবান ব্রহ্ম ##ের সঙ্গে তাঁর তপস্যা করেছিলেন কিন্তু স্বীকার করেছিলেন যে তিনি বলেছেন যে তাঁর দোষ ##কেই বিয়ে করতে পারতেন ৷"
177041	"মনে হয় ছাতির সাথে চোখের সমস্যা হলে আরাম পাওয়া যায় আর একসঙ্গে আনার জন্য রোজ একবার আরাম পাওয়া যায় ৷"
177042	"ভোর ##বেলা আমরা তাকে বলে যে বাচ্চার যদি শিশু তার ঠোঁট ##ে এসে এসে মুক্তোর সাথে ঝ ##িম করে এবং চমক এসে পৌঁছয় ##নি এবং সতর্ক করে সম্মান সতর্কতা অবলম্বন করে ।"

Pipeline finished successfully.
